# MSCSF25M026-Inshaal
Project Title:
#BipedalWalker Training with Stable-Baselines3 + Gymnasium


# Install Required Libraries


In [3]:
!pip install "stable-baselines3>=2.3.0" gymnasium[box2d] pygame imageio imageio-ffmpeg --upgrade

INFO: pip is looking at multiple versions of gymnasium[box2d] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 42.6 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


# Import Required Libraries

In [4]:
import gymnasium as gym
import os, glob
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from base64 import b64encode

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Create and Normalize BipedalWalker Environment

In [5]:
def make_env():
    env = gym.make('BipedalWalker-v3')
    return Monitor(env)

from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

env = DummyVecEnv([make_env])

env = VecNormalize(
    env,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.
)

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/

# Initialize and Train PPO Agent

In [ ]:
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=1e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.02,
    clip_range=0.2,
)
model.learn(total_timesteps=1000000)
model.save("ppo_1M")

model.learn(total_timesteps=1000000)
model.save("ppo_2M")

model.learn(total_timesteps=1000000)
model.save("ppo_3M")

model.learn(total_timesteps=1000000)
model.save("ppo_4M")

model.learn(total_timesteps=1000000)
model.save("ppo_5M")

Streaming output truncated to the last 5000 lines.
|    loss                 | -0.485       |
|    n_updates            | 21910        |
|    policy_gradient_loss | -0.00596     |
|    std                  | 132          |
|    value_loss           | 0.0424       |
------------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 991          |
|    ep_rew_mean          | 179          |
| time/                   |              |
|    fps                  | 575          |
|    iterations           | 237          |
|    time_elapsed         | 842          |
|    total_timesteps      | 485376       |
| train/                  |              |
|    approx_kl            | 0.0029172562 |
|    clip_fraction        | 0.0174       |
|    clip_range           | 0.2          |
|    entropy_loss         | -25          |
|    explained_variance   | 0.707        |
|    learning_rate        | 0.0001       |
|  

# Save Environment Normalization Parameters

In [7]:
env.save("vec_normalize.pkl")

# Save Trained PPO Model

In [8]:
model.save('ppo_bipedal')

# Load Normalized Environment for Evaluation

In [9]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# normal env
eval_env = gym.make("BipedalWalker-v3", render_mode="rgb_array")

# load normalization
temp_env = DummyVecEnv([make_env])
temp_env = VecNormalize.load("vec_normalize.pkl", temp_env)

temp_env.training = False
temp_env.norm_reward = False

# Evaluate PPO Agent and Record Performance Video

In [10]:
import gymnasium as gym
import os
import shutil
from gymnasium.wrappers import RecordVideo

# delete old videos
if os.path.exists("videos"):
    shutil.rmtree("videos")

video_folder = "videos"
os.makedirs(video_folder, exist_ok=True)

# create env
eval_env = gym.make(
    "BipedalWalker-v3",
    render_mode="rgb_array"
)

# record wrapper
eval_env = RecordVideo(
    eval_env,
    video_folder=video_folder,
    episode_trigger=lambda e: True
)

obs, _ = eval_env.reset()

for step in range(5000):

    # IMPORTANT normalization
    obs_norm = temp_env.normalize_obs(obs)

    action, _ = model.predict(
        obs_norm,
        deterministic=True
    )

    obs, reward, terminated, truncated, _ = eval_env.step(action)

    if terminated or truncated:
        print("Episode finished")
        break

# VERY IMPORTANT
eval_env.close()

print("Video saved")

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode finished


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Video saved


# Display Recorded PPO Agent Video

In [11]:
import glob
from IPython.display import Video

video_files = glob.glob("videos/*.mp4")

print(video_files)

Video(video_files[0], embed=True)

['videos/rl-video-episode-0.mp4']
